In [3]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [4]:
from huggingface_hub import snapshot_download

snapshot_download(repo_id="momina02/OutteTTS-urdu-dataset", 
                  repo_type="dataset", local_dir="./OutteTTS-urdu-dataset")

Fetching 13 files: 100%|██████████| 13/13 [00:00<00:00, 13119.82it/s]


'/home/ubuntu/OutteTTS-urdu-dataset'

In [6]:
files = glob('OutteTTS-urdu-dataset/*/*.parquet')
len(files)

9

In [10]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in tqdm(files):
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in range(len(df)):
            t = df['text'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}_{df['speaker_name'].iloc[i]}"
            })
        
    return data

In [8]:
data = loop((files[:1], 0))

100%|██████████| 1/1 [00:00<00:00,  4.03it/s]


In [11]:
data = multiprocessing(files, loop, len(files))

100%|██████████| 1/1 [01:27<00:00, 87.65s/it]


In [12]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'OutteTTS-urdu-dataset_audio/OutteTTS-urdu-dataset-data-train-00001-of-00008_0.mp3',
 'text': 'انہیں فوج نے ساتھ ملایا اور مراد دی',
 'speaker': 'OutteTTS-urdu-dataset_audio_uat_speaker'}

In [13]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'OutteTTS-urdu-dataset')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 106.01ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████| 1.26MB / 1.26MB,  135kB/s  
Processing Files (1 / 1): 100%|██████████| 1.26MB / 1.26MB,  134kB/s  
New Data Upload: 100%|██████████| 1.26MB / 1.26MB,  134kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:09<00:00,  9.75s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/c243fc23d3a1b9d6f48210a640ab8d5e00c60dee', commit_message='Upload dataset', commit_description='', oid='c243fc23d3a1b9d6f48210a640ab8d5e00c60dee', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [14]:
audio_files = [d['audio_filename'] for d in data]

with open('OutteTTS-urdu-dataset-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [17]:
# !zip -rq OutteTTS-urdu-dataset_audio.zip OutteTTS-urdu-dataset_audio

In [18]:
# !hf upload malaysia-ai/Multilingual-TTS OutteTTS-urdu-dataset_audio.zip --repo-type=dataset

In [20]:
# !zip -rq OutteTTS-urdu-dataset_audio_neucodec.zip OutteTTS-urdu-dataset_audio_neucodec

In [22]:
# !hf upload malaysia-ai/Multilingual-TTS OutteTTS-urdu-dataset_audio_neucodec.zip --repo-type=dataset